# Phase 6 — Opportunity/RFP analysis and visible filter proposals

## Goal

Convert one RFP, AMI, terms of reference, or commercial need into structured,
reviewable requirements and business-filter proposals **before** retrieval and
RAG. The notebook is resumable, hash-pinned, and safe by default.

This phase makes **zero external LLM calls**, never modifies the source corpus,
never auto-confirms business filters, and preserves the pending Phase 5.1 expert
evaluation gate.

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile

PROJECT_FOLDER_NAME = "Devoteam_AI_CLEAN_PIPELINE"
PROJECT_PARENT_FOLDER_NAME = "Devoteam internship"
PACKAGE_FILENAME = "PHASE_6_OPPORTUNITY_ANALYSIS_PACKAGE.zip"
PACKAGE_SHA256 = "1c8e8ea1e1a9bc80536d63389625b53d994143ccfd991edc38e9a4555d44c6fd"
PACKAGE_MANIFEST_SHA256 = "cbf65c473fa45511e2adaa981f335b8e952342aefa1575232a4c80f0e77070d7"
SNAPSHOT_ID = "20260714T154731Z_129ff982c8"
PHASE4_RUN_NAME = "phase4_corpus_v1"
PHASE5_RUN_NAME = "phase5_hybrid_retrieval_v1"

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

print("Phase 6 contract loaded. External LLM calls are disabled.")

## 1. Mount Drive and locate the clean project

Colab discovers the project from `config/project.yaml`. Local validation may
set `DEVOTEAM_PROJECT_ROOT` and `DEVOTEAM_PHASE6_PACKAGE`.

In [ ]:
override = os.environ.get("DEVOTEAM_PROJECT_ROOT")
if override:
    PROJECT_ROOT = Path(override).resolve()
else:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = (
        Path("/content/drive/MyDrive")
        / PROJECT_PARENT_FOLDER_NAME
        / PROJECT_FOLDER_NAME
    ).resolve()
    assert PROJECT_ROOT.is_dir(), f"Clean project folder not found: {PROJECT_ROOT}"
    assert (PROJECT_ROOT / "config" / "project.yaml").exists(), (
        f"Project configuration not found: {PROJECT_ROOT / 'config' / 'project.yaml'}"
    )

PACKAGE_PATH = Path(os.environ.get("DEVOTEAM_PHASE6_PACKAGE", PROJECT_ROOT / PACKAGE_FILENAME)).resolve()
assert PROJECT_ROOT.name == PROJECT_FOLDER_NAME, PROJECT_ROOT
assert PACKAGE_PATH.exists(), f"Missing package: {PACKAGE_PATH}"
print(f"Project root: {PROJECT_ROOT}")

## 2. Verify and install the signed package

Only the six manifest-listed Phase 6 files may be installed. Existing identical
files are skipped; conflicting files stop the run.

In [ ]:
assert file_sha256(PACKAGE_PATH) == PACKAGE_SHA256, "Phase 6 package hash mismatch"
with zipfile.ZipFile(PACKAGE_PATH) as archive:
    names = archive.namelist()
    assert "PHASE_6_PACKAGE_MANIFEST.json" in names
    manifest_bytes = archive.read("PHASE_6_PACKAGE_MANIFEST.json")
    assert hashlib.sha256(manifest_bytes).hexdigest() == PACKAGE_MANIFEST_SHA256
    package_manifest = json.loads(manifest_bytes)
    allowed = set(package_manifest["files"]) | {"PHASE_6_PACKAGE_MANIFEST.json"}
    assert set(names) == allowed, "Package contains undeclared files"
    installed = skipped = 0
    for name in names:
        target = (PROJECT_ROOT / name).resolve()
        assert target == PROJECT_ROOT or PROJECT_ROOT in target.parents, name
        data = archive.read(name)
        if name != "PHASE_6_PACKAGE_MANIFEST.json":
            assert hashlib.sha256(data).hexdigest() == package_manifest["files"][name]["sha256"]
        if target.exists():
            assert target.read_bytes() == data, f"Conflicting existing Phase 6 file: {name}"
            skipped += 1
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(data)
            installed += 1

requirements = PROJECT_ROOT / "requirements" / "phase6.txt"
if os.environ.get("DEVOTEAM_SKIP_PIP") != "1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)

environment = os.environ.copy()
environment["PYTHONPATH"] = str(PROJECT_ROOT / "src") + os.pathsep + environment.get("PYTHONPATH", "")
tests = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", str(PROJECT_ROOT / "tests")],
    cwd=PROJECT_ROOT,
    env=environment,
    text=True,
    capture_output=True,
)
print(tests.stdout[-4000:])
assert tests.returncode == 0, tests.stderr[-4000:]
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"Package verified: installed={installed}, unchanged={skipped}")

## 3. Verify Phase 4 filters and the Phase 5 technical index

Phase 6 is pinned to the same immutable snapshot and signed retrieval index.
The pending expert gold set remains a promotion blocker, not a technical-build blocker.

In [ ]:
from devoteam_reference_ai.phase6_opportunity import (
    create_redacted_sample,
    load_phase6_config,
    verify_phase5_dependency,
    write_analysis_run,
)

CONFIG = load_phase6_config(PROJECT_ROOT / "config" / "phase6_opportunity.yaml")
PHASE4_ROOT = PROJECT_ROOT / "data" / "canonical" / SNAPSHOT_ID / PHASE4_RUN_NAME
FILTER_VALUES = PHASE4_ROOT / "filter_values.json"
PHASE5_ROOT = PROJECT_ROOT / "data" / "indexes" / SNAPSHOT_ID / PHASE5_RUN_NAME
assert FILTER_VALUES.exists(), FILTER_VALUES
dependency = verify_phase5_dependency(PHASE5_ROOT, CONFIG)
assert file_sha256(FILTER_VALUES) == CONFIG["input"]["expected_filter_values_sha256"]
print(f"Phase 5 status: {dependency['manifest']['status']}")
print(f"Expert evaluation: {dependency['manifest']['expert_gold_set_status']}")

## 4. Select the opportunity input

Place exactly one real file named `OPPORTUNITY_INPUT.txt`, `.md`, `.pdf`, or
`.docx` under `human_inputs/phase6/`. If none exists, the notebook runs a safe
redacted sample so the technical phase can be validated without confidential data.

In [ ]:
INPUT_DIR = PROJECT_ROOT / "human_inputs" / "phase6"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
supported = set(CONFIG["input"]["supported_extensions"])
real_inputs = sorted(
    path for path in INPUT_DIR.glob("OPPORTUNITY_INPUT.*")
    if path.suffix.casefold() in supported
)
assert len(real_inputs) <= 1, f"Keep exactly one OPPORTUNITY_INPUT file: {real_inputs}"
if real_inputs:
    INPUT_PATH = real_inputs[0]
    INPUT_MODE = "REAL_USER_PROVIDED"
else:
    INPUT_PATH = INPUT_DIR / "OPPORTUNITY_SAMPLE_REDACTED.txt"
    create_redacted_sample(INPUT_PATH)
    INPUT_MODE = "SAMPLE_REDACTED"
assert INPUT_DIR.resolve() in INPUT_PATH.resolve().parents
print(f"Input mode: {INPUT_MODE}")
print(f"Input file: {INPUT_PATH.name}")

## 5. Analyze, compile visible filters, and sign the output

The output contains proposals only. No business filter is automatically applied
and the original opportunity file is not copied.

In [ ]:
OUTPUT_ROOT = PROJECT_ROOT / CONFIG["output"]["root"]
RUN_ROOT, MANIFEST = write_analysis_run(
    input_path=INPUT_PATH,
    filter_values_path=FILTER_VALUES,
    output_root=OUTPUT_ROOT,
    config=CONFIG,
    input_mode=INPUT_MODE,
)

print("PHASE 6 OPPORTUNITY ANALYSIS: PASS")
print(f"Status: {MANIFEST['status']}")
print(f"Requirements: {MANIFEST['requirements']}")
print(f"Visible filter proposals: {MANIFEST['filter_proposals']}")
print("Business filters auto-applied: 0")
print("External LLM calls: 0")
print(f"Output: {RUN_ROOT}")
if INPUT_MODE == "SAMPLE_REDACTED":
    print("IMPORTANT: This validates the pipeline only. Add OPPORTUNITY_INPUT.* later for a real analysis.")

## Next step

The technical gate is complete when the last cell prints `PASS`. For a real
opportunity, a business user reviews `OPPORTUNITY_REVIEW.xlsx`. The next
engineering phase consumes only approved requirements and filters, runs secure
retrieval/reranking, and creates evidence-backed recommendations. Phase 5.1
expert evaluation continues separately when Devoteam assigns reviewers.